# **Model Initialization and Reference Vocabulary Generation**

This notebook serves as the foundational setup phase for our Out-of-Distribution (OOD) detection pipeline. Here, we demonstrate the end-to-end process of preparing the feature extraction backbone and constructing the reference database.

### **Objectives**
1. **Model Instantiation:** Download and initialize Meta's pre-trained **DINOv3** model, which serves as our primary dense feature extractor.
2. **Dataset Preparation:** Acquire the base dataset and systematically inject Out-of-Distribution (OOD) labels to establish our experimental ground truth.
3. **Histogram Generation:** Process the dataset to extract spatial features and compute Bag-of-Visual-Words (BoVW) histograms for each image.
4. **Serialization:** Save the extracted reference data and vocabulary models for downstream evaluation.

> **Methodological Note:** Because K-Nearest Neighbors (K-NN) is a *lazy learning* algorithm, this phase does not involve backpropagation or updating neural network weights. Instead of traditional "training," we are constructing a comprehensive reference vocabulary and populating a database of feature histograms that the K-NN algorithm will query during inference.

**Integration Context:** The serialized artifacts generated in this notebook are explicitly designed to be imported into our real-time interactive camera application (`demo.py`), enabling live OOD detection demonstrations.

## **1. Environment Preparation**

 We want to dynamically configure our hardware device to utilize GPU acceleration (CUDA or MPS) if available, which is critical for efficiently extracting features using the DINOv3 Vision Transformer.

### **Imports**

In [ ]:
import torch
import numpy as np
import torchvision
import matplotlib.pyplot as plt
import kagglehub
from transformers import AutoImageProcessor, AutoModel
import os
import torchvision.transforms as transforms

### **Cuda**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### **Model Import**

### **1.1 DINOv3 Model Instantiation via Hugging Face**

For our feature extraction backbone, we utilize Meta's **DINOv3** architecture. The pre-trained model weights are accessed via the Hugging Face `transformers` library.

When replicating these experiments, it is strictly advised to select a model variant (e.g., ViT-Small, ViT-Base, or ViT-Large) that aligns with your available GPU VRAM and hardware constraints. Our baseline experiments default to the `vitb16` variant (Vision Transformer Base with a 16x16 patch size).

> **Resource Link:** Official model access, technical documentation, and alternative parameter sizes can be found on the [Meta DINOv3 Hugging Face Repository](https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m).

In [ ]:
# Change the model name to a model you have opted for
model_name = "facebook/dinov3-vit7b16-pretrain-lvd1689m"

In [ ]:
if 'dtype' not in locals():
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

### **1.1 Log In**

In [ ]:
from huggingface_hub import login
login()

### **1.2 Model Download**

In [ ]:
MODEL_ID = "facebook/dinov3-vitb16-pretrain-lvd1689m"

In [ ]:
# Initialize the image processor and the backbone model
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID)


In [ ]:
# Configure the model to automatically output all intermediate layer representations
model.config.output_hidden_states = True

# Move to the configured hardware device and lock the weights (Evaluation Mode)
model = model.to(device)
model.eval()

print("DINOv3 model successfully loaded, locked for inference, and moved to device.")

### **1.3 Alternative: DINOv2 via PyTorch Hub**

For researchers replicating this work who may not have access to the gated DINOv3 repository, we provide a seamless fallback using Meta's publicly available **DINOv2**.

To maintain strict pipeline compatibility, the code below utilizes an Adapter Pattern. It loads the DINOv2 weights natively via `torch.hub` while wrapping the model and preprocessing transforms to perfectly mimic the Hugging Face API. This ensures that downstream feature extraction, layer indexing, and K-NN clustering function identically regardless of the selected backbone.

In [ ]:
import torchvision.transforms as T

### **2.1 Class definitions**

In [ ]:
class DINOv2ProcessorAdapter:
    """Mimics Hugging Face's AutoImageProcessor using standard torchvision transforms."""
    def __init__(self):
        self.transform = T.Compose([
            T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
            T.CenterCrop(224),
            T.ToTensor(),
            T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])

    def __call__(self, images, return_tensors="pt"):
        # Handle both single PIL images and lists of PIL images
        if not isinstance(images, list):
            images = [images]
        pixel_values = torch.stack([self.transform(img) for img in images])
        return {"pixel_values": pixel_values}

In [ ]:
class DINOv2ModelAdapter(torch.nn.Module):
    """Wraps PyTorch Hub DINOv2 to output features exactly like Hugging Face."""
    def __init__(self, model_size='dinov2_vitb14'):
        super().__init__()
        print(f"Downloading/Loading {model_size} from PyTorch Hub...")
        self.base_model = torch.hub.load('facebookresearch/dinov2', model_size)

    def forward(self, pixel_values, **kwargs):
        # Extract features from all 12 blocks natively
        features = self.base_model.get_intermediate_layers(
            pixel_values, n=12, return_class_token=True
        )

        hidden_states = []
        for patch_tokens, cls_token in features:
            # Reconstruct the sequence: [Class Token, Patch Token 1, ..., Patch Token N]
            cls_token = cls_token.unsqueeze(1)
            layer_output = torch.cat([cls_token, patch_tokens], dim=1)
            hidden_states.append(layer_output)

        # Create a mock output object to satisfy downstream HF attribute calls
        class MockHFOutput:
            def __init__(self, hs):
                self.hidden_states = hs

        return MockHFOutput(tuple(hidden_states))

### **2.2 Model Download**

In [ ]:
processor = DINOv2ProcessorAdapter()
model = DINOv2ModelAdapter('dinov2_vitb14')

# Move to device and lock weights for inference
model = model.to(device)
model.eval()

print("DINOv2 Fallback successfully loaded and configured for HF-compatibility.")

## **2. Dataset Acquisition and Stratification**

### **2.1 Baseline Dataset: Caltech-101**
For our foundational experiments and baseline reference vocabulary, we default to the **Caltech-101** dataset. This dataset provides a robust variety of object categories, making it an ideal candidate for testing semantic clustering and establishing In-Distribution (ID) bounds.

To ensure our K-Nearest Neighbors (K-NN) classifier has a balanced reference database, we perform a **stratified 80/20 split**.
* **80% (Reference Set):** Used strictly to construct the Bag-of-Visual-Words (BoVW) vocabulary and populate the K-NN database.
* **20% (Query Set):** Held out for evaluating ID classification accuracy before we introduce Out-of-Distribution (OOD) data.

In [ ]:
from sklearn.model_selection import train_test_split

### **Caltech**

In [ ]:
caltech_full = torchvision.datasets.Caltech101(
    root='./data',
    download=True
)

In [ ]:
total_images = len(caltech_full)
categories = caltech_full.categories
print(f"Dataset successfully loaded: {total_images} high-resolution images across {len(categories)} categories.")

In [ ]:
# Extract numerical labels and generate image indices
# Note: torchvision's Caltech101 stores numerical target labels in the '.y' attribute
all_labels = caltech_full.y
all_indices = np.arange(total_images)

In [ ]:
# Perform a stratified split to maintain class distribution in both sets
train_idx, test_idx, train_labels, test_labels = train_test_split(
    all_indices,
    all_labels,
    test_size=0.20,
    random_state=42,       # Fixed seed for thesis reproducibility
    stratify=all_labels
)

### **2.2 Custom Dataset Integration (Optional)**

The pipeline is designed to be dataset-agnostic. If you are applying this hierarchical OOD detection algorithm to a specific domain (e.g., medical imaging, industrial defect detection, or a custom camera feed), you can overwrite the `train_idx`, `test_idx`, `train_labels`, and `test_labels` variables here.

We recommend using `torchvision.datasets.ImageFolder` for custom local datasets.

In [ ]:
# TODO! Fill in with dataset to download

## **3. Feature Extraction**

When passing an image through the DINOv3 Vision Transformer, the input image is divided into a grid of non-overlapping patches. The model processes these patches and outputs a sequence of high-dimensional embeddings.

Mathematically, for a single image, the final hidden state $Z$ is a tensor of shape $\mathbb{R}^{L \times D}$, where $D$ is the embedding dimension and $L$ is the total sequence length defined as:
$$L = 1 + N_{reg} + N_{patches}$$

Where:
* **$1$**: The prepended Global Class Token (`[CLS]`).
* **$N_{reg}$**: The Register Tokens introduced to absorb localized artifacts and stabilize attention.
* **$N_{patches}$**: The actual spatial image patches (e.g., $14 \times 14 = 196$ patches).

Because our OOD detection relies on localized semantic features (Bag-of-Visual-Words), we must discard the global `[CLS]` token and the register tokens, isolating only the $N_{patches}$ spatial embeddings:
$$Z_{patches} = Z[1 + N_{reg}:, :]$$

The following function processes our dataset in batches, extracts these localized spatial features, and serializes them to disk using half-precision (`float16`) to optimize storage.

In [ ]:
import os
from tqdm import tqdm

### **3.1 Method defition**

In [ ]:
def extract_and_save_features(indices, labels, dataset, split_name):
    """
    Extracts dense patch features from a dataset using the pre-loaded DINO backbone
    and serializes them to disk.

    Args:
        indices (iterable): An array of dataset indices to process (e.g., train_idx or test_idx).
        labels (iterable): The corresponding ground truth labels for the indices.
        dataset (torch.utils.data.Dataset): The source image dataset (e.g., Caltech101).
        split_name (str): Identifier for the dataset split (e.g., 'train', 'test', 'ood').

    Side Effects:
        Saves a serialized PyTorch dictionary (.pt) containing 'features' and 'labels'
        to the current working directory.
    """
    features_list = []
    labels_list = []

    # Disable gradient calculation for purely forward-pass inference
    with torch.no_grad():
        for idx, label in tqdm(zip(indices, labels), total=len(indices), desc=f"Extracting {split_name.upper()}"):

            # Retrieve image and ensure RGB color space to prevent channel mismatch errors
            img, _ = dataset[idx]
            img = img.convert("RGB")

            # Preprocess image and transfer to the target hardware accelerator
            inputs = processor(images=img, return_tensors="pt").to(device)

            # Execute forward pass through the DINOv3 backbone
            outputs = model(**inputs)

            # Isolate the final layer's hidden states
            hidden_states = outputs.last_hidden_state

            # Slice the sequence to drop the [CLS] token and Register tokens, keeping only spatial patches
            # Assuming `num_registers` is defined globally based on the specific DINO architecture
            patch_features = hidden_states[0, 1 + num_registers : ]

            # Transfer features back to CPU RAM and cast to float16 to conserve storage
            features_list.append(patch_features.cpu().to(torch.float16))
            labels_list.append(label)

    # Serialize and export the extracted features and corresponding labels
    filename = f"caltech101_{split_name}_flagship_features.pt"
    save_dict = {
        "features": features_list,
        "labels": labels_list
    }

    torch.save(save_dict, filename)
    print(f"Successfully saved {len(features_list)} feature tensors to {filename}")

### **3.2 Extraction**

In [ ]:
extract_and_save_features(train_idx, train_labels, caltech_full, "train")

In [ ]:
extract_and_save_features(test_idx, test_labels, caltech_full, "test")

### **3.3 Loading the data**

In [ ]:
train_data = torch.load("caltech101_train_flagship_features.pt", weights_only=False)
test_data = torch.load("caltech101_test_flagship_features.pt", weights_only=False)

In [ ]:
caltech_train_features = [tensor.to(torch.float32).numpy() for tensor in train_data["features"]]
caltech_train_labels = train_data["labels"]

In [ ]:
caltech_test_features = [tensor.to(torch.float32).numpy() for tensor in test_data["features"]]
caltech_test_labels = test_data["labels"]

## **4. Semantic Patch Clustering**

To construct our BoVW reference vocabulary, we must group the continuous, high-dimensional patch embeddings extracted by DINOv3 into a discrete set of semantic "visual words."

For this task, we utilize **[FAISS](https://ai.meta.com/tools/faiss/) (Facebook AI Similarity Search)**, an open-source library developed by Meta. We specifically chose FAISS over traditional clustering libraries (such as `scikit-learn`) because of its unparalleled optimization for dense vector similarity search and clustering. Given the sheer volume of our extracted features (e.g., thousands of images $\times$ 196 patches per image $\times$ embedding dimension), FAISS provides the necessary GPU acceleration and memory management to perform large-scale $K$-Means clustering efficiently.

> **Prerequisite:** Ensure FAISS is installed in your environment before proceeding.

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import time

### **Function definition**

In [ ]:
def to_numpy_array(data):
    """
    Safely converts varying data structures (PyTorch Tensors, Lists of Tensors,
    or raw lists) into a standardized NumPy array for FAISS processing.
    """
    if torch.is_tensor(data):
        return data.cpu().numpy()
    elif isinstance(data, list):
        if torch.is_tensor(data[0]):
            return torch.stack(data).cpu().numpy()
        return np.array(data)
    return data

In [ ]:
def flatten_labels(labels):
    """
    Flattens hierarchical or batched label structures into a 1D NumPy array,
    ensuring dimensional alignment with the extracted features.
    """
    if torch.is_tensor(labels):
        return labels.cpu().numpy().flatten()
    elif isinstance(labels, list):
        if torch.is_tensor(labels[0]):
            return torch.tensor(labels).cpu().numpy().flatten()
        return np.array(labels).flatten()
    return labels

### **Data Prep**

In [ ]:
# Convert the serialized PyTorch features into NumPy arrays
train_feats_np = to_numpy_array(caltech_train_features)
test_feats_np = to_numpy_array(caltech_test_features)

In [ ]:
# ---------------------------------------------------------
# Tensor Reshaping for Clustering
# ---------------------------------------------------------
# Original shape: (num_images, num_patches, embedding_dim)
# Target shape:   (num_images * num_patches, embedding_dim)
# We flatten the spatial dimension because K-Means treats every single
# patch as an independent vector in the latent space.
flat_train = train_feats_np.reshape(-1, train_feats_np.shape[-1]).astype(np.float32)

### **4.2 K-Means Vocabulary Generation**

We define our vocabulary size ($K$) as the number of cluster centroids. In this baseline experiment, we set $K = 1024$. This hyperparameter determines the granularity of our semantic dictionary; a higher $K$ captures finer visual details, while a lower $K$ generalizes broader semantic concepts.

In [ ]:
# Define vocabulary size (K) and vector dimensionality (d)
k = 1024
d = flat_train.shape[1]

In [ ]:
print(f"Training FAISS K-Means with K={k} clusters...")
start_time = time.time()

# Dynamically check for GPU availability to accelerate clustering
use_gpu = faiss.get_num_gpus() > 0

# Initialize the FAISS K-Means algorithm
# niter=20 is chosen as a balance between convergence and computational time
kmeans = faiss.Kmeans(d, k, niter=20, verbose=True, gpu=use_gpu)

# Execute the clustering algorithm on the flattened training patches
kmeans.train(flat_train)

In [ ]:
# Initialize an L2 distance index and populate it with the learned centroids.
# This index will be used later to quickly assign new patches to their nearest visual word.
index = faiss.IndexFlatL2(d)
index.add(kmeans.centroids)

In [ ]:
elapsed_time = time.time() - start_time
print(f"Clustering completed in {elapsed_time:.2f} seconds.")
print(f"Successfully indexed {index.ntotal} cluster centroids.")

## **5. Bag-of-Visual-Words Histogram Construction**

With our semantic vocabulary established, we can now translate the continuous patch embeddings of each image into a discrete, globally recognizable format.

For every image in our dataset, we map each of its constituent patch features to the nearest cluster centroid (Visual Word) in our FAISS index. We then tally these assignments to create a frequency histogram of length $K$.

### **L1 Normalization**
To ensure our distance metrics (such as Chi-Square) function correctly during K-NN classification, we must convert these raw frequency counts into probability distributions. We achieve this by applying **$L_1$ Normalization** to each histogram, ensuring the sum of all bins equals 1.

Mathematically, for a histogram $H$ with bins $c$, the normalized bin value $h_i$ is computed as:
$$h_i = \frac{c_i}{\sum_{j=1}^{K} c_j + \epsilon}$$
*(where $\epsilon = 1\times 10^{-10}$ is a small constant added for numerical stability to prevent division by zero).*

### **5.1 Function Definition**

In [ ]:
def build_histograms(features_array, faiss_index, vocab_size, emb_dim):
    """
    Constructs Bag-of-Visual-Words (BoVW) histograms for a collection of images.

    Args:
        features_array (np.ndarray): Array of shape (num_images, num_patches, emb_dim).
        faiss_index (faiss.Index): The trained FAISS index containing the cluster centroids.
        vocab_size (int): The number of visual words (K clusters).
        emb_dim (int): The dimensionality of each patch feature.

    Returns:
        np.ndarray: An array of raw frequency histograms of shape (num_images, vocab_size).
    """
    hists = []

    # Iterate through each image's features with a progress bar
    for feats in tqdm(features_array, desc="Building Histograms"):
        # Flatten the spatial dimension to process patches individually
        feats_f32 = feats.reshape(-1, emb_dim).astype(np.float32)

        # Search the FAISS index for the single nearest centroid (k=1) for each patch
        # 'I' contains the indices of the nearest visual words
        _, nearest_centroid_indices = faiss_index.search(feats_f32, 1)

        # Count the frequency of each visual word to build the histogram
        hist = np.bincount(nearest_centroid_indices.flatten(), minlength=vocab_size)
        hists.append(hist)

    return np.array(hists)

### **5.2 Creating them**

In [ ]:
print("Constructing reference histograms (Training Set)...")
train_hists = build_histograms(train_feats_np, index, k, d)

# Apply L1 Normalization to the Training histograms
train_l1 = train_hists / (np.sum(train_hists, axis=1, keepdims=True) + 1e-10)

In [ ]:
print("Constructing query histograms (Testing Set)...")
test_hists = build_histograms(test_feats_np, index, k, d)

# Apply L1 Normalization to the Testing histograms
test_l1 = test_hists / (np.sum(test_hists, axis=1, keepdims=True) + 1e-10)

## **6. Semantic Label Mapping and Final Serialization**

To ensure our downstream K-Nearest Neighbors classifier outputs interpretable, human-readable predictions, we must map the numerical target indices back to their semantic string equivalents (e.g., mapping class `0` to `"accordion"`).

Finally, we serialize these mapped labels alongside our $L_1$-normalized histograms into a compressed `.npz` archive. This archive acts as the  "Reference Database" for our feature space and can be seamlessly loaded into lightweight, real-time inference applications.

In [ ]:
categories = caltech_full.categories
print(f"Dataset Vocabulary Sample: {categories[:5]}")

In [ ]:
integer_labels = flatten_labels(train_labels)
integer_labels = [int(lbl) for lbl in integer_labels]

In [ ]:
# Map numerical indices to their corresponding string labels
string_labels = np.array([categories[i] for i in integer_labels])
print(f"Mapped String Labels Sample: {string_labels[:5]}")

## **7. Serialization of Model Artifacts and Configuration**

To transition from the offline generation phase to real-time inference (e.g., in our interactive `demo.py` application or downstream evaluation notebooks), we must export our pipeline's state.

We serialize three distinct artifacts:
1. **Visual Vocabulary (`.npy`):** The $K$-Means centroids used to quantize novel, unseen image patches into our established semantic space.
2. **Reference Database (`.npz`):** The highly compressed, $L_1$-normalized histograms and their corresponding ground-truth labels.
3. **Inference Configuration (`.json`):** A standard configuration file containing the architectural hyperparameters and detection thresholds (such as distance and chaos thresholds) governing our Hierarchical OOD logic.

In [ ]:
import json

In [ ]:
output_dir = "./model_artifacts"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Save the FAISS Visual Vocabulary (Centroids)
centroids_path = os.path.join(output_dir, f"faiss_dictionary_k{k}.npy")

np.save(centroids_path, kmeans.centroids)

In [ ]:
# Save the K-NN Reference Database
database_path = os.path.join(output_dir, f"knn_reference_database_k{k}.npz")

np.savez_compressed(
    database_path,
    histograms=train_l1,
    labels=flatten_labels(caltech_train_labels)
)

In [ ]:
#Define and Save the Inference Configuration

hyperparameters = {
    "K_NN": 7,
    "DISTANCE_THRESHOLD": 0.5,
    "CHAOS_THRESHOLD": 3,
    "OOD_LABEL": -1,
    "num_clusters": k,
    "feature_dim": d
}

config_path = os.path.join(output_dir, "model_config.json")

# Use indent=4 to make the JSON file human-readable for debugging
with open(config_path, 'w') as f:
    json.dump(hyperparameters, f, indent=4)